# Flood summaries for ThinkHazard

This script performs flood hazard ranking by administrative unit using global-extent
Fathom tiles hosted on AWS S3, rather than country-extent locally downloaded data.

The hazard ranking is based on:
- Value threshold: Minimum flood depth (cm) to consider
- Area threshold: Minimum percentage of area affected
- Hazard score: Count of return periods meeting both thresholds (0-3)


In [1]:
import os, time, io, json, sys
import urllib3
import boto3

import geopandas as gpd
import pandas as pd
import numpy as np

from functools import reduce
from urllib3.exceptions import InsecureRequestWarning
from botocore import UNSIGNED
from botocore.config import Config
from tqdm.notebook import tqdm

# Import helper functions
from gfdrr_helper import *

urllib3.disable_warnings(InsecureRequestWarning)

def tPrint(s):
    """prints the time along with the message"""
    print("%s\t%s" % (time.strftime("%H:%M:%S"), s))

s3_client = boto3.client('s3', verify=False, config=Config(signature_version=UNSIGNED))

%load_ext autoreload
%autoreload 2

In [76]:
local_folder = "C:/WBG/Work/Projects/ThinkHazard"
out_folder = os.path.join(local_folder, "FATHOM_summaries")
map_folder = os.path.join(local_folder, "FATHOM_maps")
for tF in [out_folder, map_folder]:
    if not os.path.exists(tF):
        os.makedirs(tF)
vrt_folder = r"C:\WBG\Work\data\FATHOM"
s3_bucket = "wbg-geography01"
s3_prefix = "FATHOM"
return_periods = [10, 100, 500, 1000]
flood_files = [
    ["FU", "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-FLUVIAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"],
    ["CU", "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-COASTAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"],
    ['PD', "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-PLUVIAL-DEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"]
]

admin_boundaries_file = r"C:\WBG\Work\data\ADMIN\NEW_WB_BOUNDS\FOR_PUBLICATION\crs_4326\parquet\WB_GAD_ADM2.parquet"
inA = gpd.read_parquet(admin_boundaries_file)

In [77]:
cur_out_folder = os.path.join(out_folder, "FATHOM_Detailed")
if not os.path.exists(cur_out_folder):
    os.makedirs(cur_out_folder)

cur_map_folder = os.path.join(map_folder, "FATHOM_Detailed")
if not os.path.exists(cur_map_folder):
    os.makedirs(cur_map_folder)

with rasterio.Env(GDAL_HTTP_UNSAFESSL='YES'):
    for sel_country in ['TUN','GEO','PHL']: #inA['ISO_A3'].unique(): #
        all_res = []
        out_file = os.path.join(cur_out_folder, f"FATHOM_ThinkHazard_summary_{sel_country}.csv")
        sel_a = inA[inA['ISO_A3'] == sel_country]                           
        if not os.path.exists(out_file) and not (sel_country in ["FJI",'RUS']):
            tPrint(f"Processing country: {sel_country}")
            for lbl, raster_file in flood_files:
                for return_period in return_periods:
                    tPrint(f"Processing {lbl} for {return_period} year return period")
                    sel_raster_file = raster_file.format(rp=return_period)
                    sel_raster = f"s3://{s3_bucket}/{s3_prefix}/{sel_raster_file}"
                    res_a = calculate_think_hazard_score(sel_a, sel_raster, 
                                                         depth_threshold=50, idx_col='ADM2CD_c',
                                                         all_touched=True, min_val=0, max_val=10000, no_data=-32767)
                    res_a.rename(columns={'frac_area_flooded': f'frac_area_flooded_{lbl}_{return_period}yr',
                                          'mean_val': f'mean_val_{lbl}_{return_period}yr'
                                         }, inplace=True)
                    all_res.append(res_a)                    
            all_res_df = reduce(lambda left, right: pd.merge(left, right, on='ADM2CD_c', how='outer'), all_res) 
            all_res_df.to_csv(out_file, index=False)

            sel_a = inA[inA['ISO_A3'] == sel_country]                                   
            map_adm = pd.merge(sel_a, all_res_df, on='ADM2CD_c', how='left')
            map_flood(map_adm, return_period=100, out_file=os.path.join(cur_map_folder, f"flood_map_{sel_country}_100yr.png"))
        else:
            tPrint(f"File already exists for {sel_country}, skipping...")

12:41:58	Processing country: TUN
12:41:58	Processing FU for 10 year return period
12:42:50	Processing FU for 100 year return period
12:43:43	Processing FU for 500 year return period
12:44:34	Processing FU for 1000 year return period
12:45:29	Processing CU for 10 year return period
12:46:04	Processing CU for 100 year return period
12:46:44	Processing CU for 500 year return period
12:47:25	Processing CU for 1000 year return period
12:48:01	Processing PD for 10 year return period
12:49:08	Processing PD for 100 year return period
12:50:28	Processing PD for 500 year return period
12:51:44	Processing PD for 1000 year return period
12:53:15	Processing country: GEO
12:53:15	Processing FU for 10 year return period
12:53:59	Processing FU for 100 year return period
12:54:42	Processing FU for 500 year return period
12:55:43	Processing FU for 1000 year return period
12:56:16	Processing CU for 10 year return period
12:56:31	Processing CU for 100 year return period
12:56:47	Processing CU for 500 year

# DEBUGGING

In [25]:
from rasterio.windows import from_bounds
from rasterio.features import rasterize
from affine import Affine

Affine?


Init signature:
Affine(
    a: float,
    b: float,
    c: float,
    d: float,
    e: float,
    f: float,
    g: float = 0.0,
    h: float = 0.0,
    i: float = 1.0,
)
Docstring:     
Two dimensional affine transform for 2D linear mapping.

Parameters
----------
a, b, c, d, e, f : float
    Coefficients of an augmented affine transformation matrix

    | x' |   | a  b  c | | x |
    | y' | = | d  e  f | | y |
    | 1  |   | 0  0  1 | | 1 |

    `a`, `b`, and `c` are the elements of the first row of the
    matrix. `d`, `e`, and `f` are the elements of the second row.

Attributes
----------
a, b, c, d, e, f, g, h, i : float
    The coefficients of the 3x3 augmented affine transformation
    matrix

    | x' |   | a  b  c | | x |
    | y' | = | d  e  f | | y |
    | 1  |   | g  h  i | | 1 |

    `g`, `h`, and `i` are always 0, 0, and 1.

The Affine package is derived from Casey Duncan's Planar package.
See the copyright statement below.  Parallel lines are preserved by
these transforms

In [71]:
sel_country = 'TUN'
raster_path = "s3://wbg-geography01/FATHOM/FLOOD_MAP-1ARCSEC-NW_OFFSET-1in100-COASTAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"
inD = inA[inA['ISO_A3'] == sel_country].copy()
depth_threshold=50
idx_col='ADM2CD_c'
all_touched=True
min_val=0
max_val=10000
no_data=-32768

with rasterio.Env(GDAL_HTTP_UNSAFESSL='YES'):
    curRaster = rasterio.open(raster_path)
    fCount = 0
    res = {}
    nodata_value = curRaster.nodata if no_data is None else no_data
    #for idx, row in inD.iterrows():
    row = inD.loc[inD['ADM2CD_c'] == 'TUN015005'].iloc[0]
    geometry = row["geometry"]
    fCount = fCount + 1
    
    ul = curRaster.index(geometry.bounds[0], geometry.bounds[3])
    lr = curRaster.index(geometry.bounds[2], geometry.bounds[1])

    window = from_bounds(*geometry.bounds, transform=curRaster.transform)
    data = curRaster.read(1, window=window)
    data = np.where(data == nodata_value, np.nan, data)
    
    # Apply min and max value filters if provided
    if min_val is not None:
        data[data < min_val] = 0
    if max_val is not None:
        data[data > max_val] = 0

    shifted_affine = curRaster.window_transform(window)
    mask = rasterize(
        [(geometry, 0)],
        out_shape=data.shape,
        transform=shifted_affine,
        all_touched=all_touched,
        fill=1,
        dtype=np.uint8
    )
    # Add to the mask areas that are nan in the data
    mask = np.where(np.isnan(data), 1, mask)

    # create a masked numpy array
    masked_data = np.ma.array(data=data, mask=mask.astype(bool))
    
    

        
    


In [75]:
(masked_data > depth_threshold).sum()

3342

In [70]:
#Write clip to file
with rasterio.open("C:/Temp/FATHOM_TUN/TUN015005_mask.tif", 'w', driver='GTiff',
                   height=masked_data.shape[0], width=masked_data.shape[1],
                   count=1, dtype=masked_data.dtype,
                   crs=curRaster.crs, transform=shifted_affine) as dst:
    dst.write(mask, 1)

with rasterio.open("C:/Temp/FATHOM_TUN/TUN015005_data.tif", 'w', driver='GTiff',
                   height=masked_data.shape[0], width=masked_data.shape[1],
                   count=1, dtype=masked_data.dtype,
                   crs=curRaster.crs, transform=shifted_affine) as dst:
    dst.write(masked_data.filled(nodata_value), 1)

In [69]:
masked_data.data

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ...,  0.,  0.,  0.],
       [nan, nan, nan, ...,  0.,  0.,  0.],
       [nan, nan, nan, ...,  0.,  0.,  0.]])

In [60]:
masked_data.mask

array([[ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       ...,
       [ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False]])

In [59]:
data

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ...,  0.,  0.,  0.],
       [nan, nan, nan, ...,  0.,  0.,  0.],
       [nan, nan, nan, ...,  0.,  0.,  0.]])

# Extracting sample data

In [ ]:
sys.path.insert(0, "C:\\WBG\\Work\\Code\\GOSTrocks\\src")
import GOSTrocks.rasterMisc as rMisc

In [ ]:
temp_out_folder = "C:/Temp/FATHOM_TUN"
if not os.path.exists(temp_out_folder):
    os.makedirs(temp_out_folder)

sel_admin = inA.loc[inA['ISO_A3'] == "TUN"]
sel_admin.to_file(os.path.join(temp_out_folder, "TUN_admin.gpkg"), driver="GPKG")

return_periods = [10, 100, 500, 1000]
flood_files = [
    ["FU", "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-FLUVIAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"],
    ["CU", "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-COASTAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"],
    ['PD', "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-PLUVIAL-DEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"]
]

for return_period in return_periods:
    for lbl, raster_file in flood_files:
        temp_out_file = os.path.join(temp_out_folder, f"TUN_{lbl}_{return_period}yr.tif")
        if not os.path.exists(temp_out_file):
            sel_raster_file = raster_file.format(rp=return_period)
            sel_raster = f"s3://{s3_bucket}/{s3_prefix}/{sel_raster_file}"
            with rasterio.Env(GDAL_HTTP_UNSAFESSL='YES'):
                inR = rasterio.open(sel_raster)
                rMisc.clipRaster(inR, sel_admin, temp_out_file, crop=False)